# 00 — Setup Check

Verifies that the development environment is ready for the SD-A-01-RAG project.

This notebook must run end-to-end with no errors before declaring Phase 0 complete (per MASTER_PLAN.md §7, Definition of done).

It checks:

1. All required packages are installed at the pinned versions and importable.
2. System info (Python, OS, CPU, conda env) is recorded.
3. Ollama is reachable and the two required local models are present.
4. A 1-sentence test prompt against `llama3.1:8b-instruct-q4_K_M` returns a coherent answer with `temperature=0, seed=42`.
5. The corpus PDFs in `dataset/pdfs/` are present and listed.


## 1. Required packages — version + importability

In [1]:
import importlib
from importlib.metadata import version, PackageNotFoundError

# (PyPI name, import name) — `None` for meta-packages we don't import directly.
packages = [
    ("langchain",             "langchain"),
    ("langchain-community",   "langchain_community"),
    ("langchain-ollama",      "langchain_ollama"),
    ("faiss-cpu",             "faiss"),
    ("sentence-transformers", "sentence_transformers"),
    ("ollama",                "ollama"),
    ("pypdf",                 "pypdf"),
    ("tiktoken",              "tiktoken"),
    ("numpy",                 "numpy"),
    ("pandas",                "pandas"),
    ("pyarrow",               "pyarrow"),
    ("matplotlib",            "matplotlib"),
    ("seaborn",               "seaborn"),
    ("codecarbon",            "codecarbon"),
    ("psutil",                "psutil"),
    ("tqdm",                  "tqdm"),
    ("jupyter",               None),  # meta-package, no top-level module
    ("ipykernel",             "ipykernel"),
]

failures = []
print(f"{'package':<28} {'version':<12} import")
print("-" * 60)
for pip_name, import_name in packages:
    try:
        v = version(pip_name)
    except PackageNotFoundError:
        print(f"{pip_name:<28} {'MISSING':<12} ---")
        failures.append((pip_name, "not installed"))
        continue

    if import_name is None:
        print(f"{pip_name:<28} {v:<12} (meta-package)")
        continue

    try:
        importlib.import_module(import_name)
        print(f"{pip_name:<28} {v:<12} ok")
    except Exception as e:
        print(f"{pip_name:<28} {v:<12} FAILED: {e}")
        failures.append((pip_name, f"import error: {e}"))

print()
if failures:
    raise RuntimeError(f"{len(failures)} package(s) not ready: {failures}")
print("All required packages importable.")


package                      version      import
------------------------------------------------------------
langchain                    1.2.18       ok
langchain-community          0.4.1        ok
langchain-ollama             1.1.0        ok
faiss-cpu                    1.13.2       ok
sentence-transformers        5.4.1        ok
ollama                       0.6.2        ok
pypdf                        6.11.0       ok
tiktoken                     0.12.0       ok
numpy                        2.4.4        ok
pandas                       3.0.2        ok
pyarrow                      24.0.0       ok
matplotlib                   3.10.9       ok
seaborn                      0.13.2       ok
codecarbon                   3.2.6        ok
psutil                       7.2.2        ok
tqdm                         4.67.3       ok
jupyter                      1.1.1        (meta-package)
ipykernel                    7.2.0        ok

All required packages importable.


## 2. System info

In [2]:
import os, sys, platform, subprocess

print(f"Python version : {sys.version.splitlines()[0]}")
print(f"Executable     : {sys.executable}")
print(f"Platform       : {platform.platform()}")
print(f"Machine        : {platform.machine()}")
print(f"Processor      : {platform.processor() or '(unknown)'}")
print(f"CONDA_DEFAULT_ENV : {os.environ.get('CONDA_DEFAULT_ENV', '(not set)')}")
print(f"CONDA_PREFIX      : {os.environ.get('CONDA_PREFIX', '(not set)')}")

# CPU brand on macOS
try:
    cpu_brand = subprocess.check_output(
        ["sysctl", "-n", "machdep.cpu.brand_string"], text=True
    ).strip()
    print(f"CPU brand      : {cpu_brand}")
except Exception:
    pass

# RAM total on macOS
try:
    ram_bytes = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"], text=True).strip())
    print(f"RAM total      : {ram_bytes / (1024**3):.1f} GB")
except Exception:
    pass


Python version : 3.14.2 | packaged by Anaconda, Inc. | (main, Dec 19 2025, 11:42:52) [Clang 20.1.8 ]
Executable     : /opt/anaconda3/envs/dsenv/bin/python3.14
Platform       : macOS-26.4.1-arm64-arm-64bit-Mach-O
Machine        : arm64
Processor      : arm
CONDA_DEFAULT_ENV : dsenv
CONDA_PREFIX      : /opt/anaconda3/envs/dsenv
CPU brand      : Apple M4
RAM total      : 16.0 GB


## 3. Ollama — local models present + smoke-test prompt

The project uses two local LLMs:

- `llama3.1:8b-instruct-q4_K_M` (primary, ~4.9 GB)
- `phi3.5:3.8b-mini-instruct-q4_K_M` (backup, ~2.2 GB)

Both must be pulled before this notebook runs:

```bash
ollama pull llama3.1:8b-instruct-q4_K_M
ollama pull phi3.5:3.8b-mini-instruct-q4_K_M
```


In [3]:
import ollama

REQUIRED_MODELS = [
    "llama3.1:8b-instruct-q4_K_M",
    "phi3.5:3.8b-mini-instruct-q4_K_M",
]

# `ollama.list()` may return either a typed object or a dict depending on
# the client version; handle both.
result = ollama.list()
models_attr = getattr(result, "models", None)
if models_attr is None:
    models_attr = result.get("models", []) if isinstance(result, dict) else []

# Each entry may also be either an object or a dict.
def _name(m):
    return getattr(m, "model", None) or getattr(m, "name", None) \
           or (m.get("model") if isinstance(m, dict) else None) \
           or (m.get("name") if isinstance(m, dict) else None)

def _size(m):
    return getattr(m, "size", None) \
           or (m.get("size") if isinstance(m, dict) else 0) or 0

local = [(_name(m), _size(m)) for m in models_attr]

print("Local Ollama models:")
for name, size in local:
    print(f"  {name:<55}  {size / (1024**3):>5.2f} GB")

local_names = [n for n, _ in local]
missing = [m for m in REQUIRED_MODELS if m not in local_names]
if missing:
    raise RuntimeError(f"Required Ollama models missing: {missing}. Run `ollama pull <model>`.")
print("\nBoth required models are installed.")


Local Ollama models:
  llama3.1:8b-instruct-q4_K_M                               4.58 GB
  phi3.5:3.8b-mini-instruct-q4_K_M                          2.23 GB

Both required models are installed.


In [4]:
# Deterministic 1-sentence test prompt against the primary LLM.
import ollama

response = ollama.chat(
    model="llama3.1:8b-instruct-q4_K_M",
    messages=[
        {"role": "user", "content": "Reply with exactly one short sentence: say hello from RAG."}
    ],
    options={"temperature": 0, "seed": 42},
)
text = response["message"]["content"].strip() if isinstance(response, dict) \
       else response.message.content.strip()

print("LLM reply:", text)
assert len(text) > 0, "Empty LLM response"
print("\nOllama smoke-test passed.")


LLM reply: Hello from RAG!

Ollama smoke-test passed.


## 4. Corpus PDFs present

In [5]:
from pathlib import Path

# This notebook lives in `notebooks/`; project root is the parent.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pdf_dir = project_root / "dataset" / "pdfs"

EXPECTED_PDFS = [
    "01_BLOOM_Luccioni_2022.pdf",
    "02_IEA_Energy_and_AI_2025.pdf",
    "03_Google_Env_2025.pdf",
    "04_EPRI_Powering_Intelligence_2024.pdf",
    "05_Greenpeace_Chipping_Point_2025.pdf",
]

print(f"Project root : {project_root}")
print(f"PDF directory: {pdf_dir}\n")

found = sorted(p.name for p in pdf_dir.glob("*.pdf"))
for p in pdf_dir.glob("*.pdf"):
    size_mb = p.stat().st_size / (1024**2)
    print(f"  {p.name:<45}  {size_mb:>7.2f} MB")

missing = [n for n in EXPECTED_PDFS if n not in found]
if missing:
    raise RuntimeError(f"Missing expected PDFs: {missing}")
print(f"\nAll {len(EXPECTED_PDFS)} expected PDFs present.")


Project root : /Users/ismaildemircan/Desktop/SD-A-01-RAG
PDF directory: /Users/ismaildemircan/Desktop/SD-A-01-RAG/dataset/pdfs

  03_Google_Env_2025.pdf                           18.55 MB
  01_BLOOM_Luccioni_2022.pdf                        0.24 MB
  05_Greenpeace_Chipping_Point_2025.pdf             2.44 MB
  02_IEA_Energy_and_AI_2025.pdf                     7.83 MB
  04_EPRI_Powering_Intelligence_2024.pdf            3.25 MB

All 5 expected PDFs present.


## Done

If every cell above ran without raising, Phase 0 is verified. Update §4 of `MASTER_PLAN.md` (Phase 0 → ✅) and append a Phase 0 entry to `PROGRESS_LOG.md`.
